In [14]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable


In [15]:
import numpy as np
import pandas as pd
import re
import string
import pickle



In [16]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text


In [17]:
# Train a simple model for demonstration
# In production, load the pre-trained model
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Load training data
data = pd.read_csv('artifacts/sentiment_analysis.csv')

# Simple preprocessing for training
from collections import Counter
vocab_counter = Counter()
for sentence in data['tweet']:
    vocab_counter.update(sentence.split())
training_tokens = [key for key in vocab_counter if vocab_counter[key] > 10]

# Vectorize training data
def vectorizer_train(ds, vocabulary):
    vectorized_lst = []
    for sentence in ds:
        sentence_lst = np.zeros(len(vocabulary))
        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_lst[i] = 1
        vectorized_lst.append(sentence_lst)
    return np.asarray(vectorized_lst, dtype=np.float32)

X_train = vectorizer_train(data['tweet'], training_tokens)
y_train = data['label']

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [18]:
with open('static/model/corpora/stopwords/english','r') as file:
    sw = file.read().splitlines()

In [19]:
# Use the same tokens from training
tokens = training_tokens

In [20]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [21]:

def preprocessing(text):
    data = pd.DataFrame([text], columns = ['tweet'])
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x,flags=re.MULTILINE) for x in x.split()))
    data["tweet"] = data["tweet"].apply(remove_punctuations)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]




In [22]:
def vectorizer(ds, vocabulary):
    vectorized_lst = []
    
    for sentence in ds:
        sentence_lst = np.zeros(len(vocabulary))
        
        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_lst[i] = 1
                
        vectorized_lst.append(sentence_lst)
        
    vectorized_lst_new = np.asarray(vectorized_lst, dtype=np.float32)
    
    return vectorized_lst_new

In [23]:
def get_prediction(vectorized_text):
    prediction = model.predict(vectorized_text)
    if prediction == 1:
        return 'negative'
    else:
        return 'positive'


In [27]:
txt = "too bad product. i hate it"
preprocessed_txt = preprocessing(txt)
vectorized_txt = vectorizer(preprocessed_txt, tokens)
prediction = get_prediction(vectorized_txt)
print(prediction)


negative
